In [1]:
import numpy as np

In [2]:
data=[
    "I love this movie",
    "This movie is amazing",
    "I hate this movie",
    "The story was boring",
    "I loved It"
]

<h2>Tokenization</h2>

In [3]:
tokens=[]

In [4]:
for d in data:
    tokens.append(d.lower().split(" "))

In [5]:
tokens

[['i', 'love', 'this', 'movie'],
 ['this', 'movie', 'is', 'amazing'],
 ['i', 'hate', 'this', 'movie'],
 ['the', 'story', 'was', 'boring'],
 ['i', 'loved', 'it']]

<h2>Created Vocabulary Set</h2>

In [6]:
vocabularySet=set()

In [7]:
for data in tokens:
    for d in data:
        vocabularySet.add(d)

In [8]:
vocabularySet

{'amazing',
 'boring',
 'hate',
 'i',
 'is',
 'it',
 'love',
 'loved',
 'movie',
 'story',
 'the',
 'this',
 'was'}

In [9]:
len(vocabularySet)

13

<h2>Vocabulary Indexing</h2>

In [10]:
words_to_id={}

In [11]:
for index,data in enumerate(vocabularySet):
    words_to_id[data]=index+1

In [12]:
words_to_id

{'i': 1,
 'love': 2,
 'it': 3,
 'amazing': 4,
 'the': 5,
 'this': 6,
 'was': 7,
 'is': 8,
 'boring': 9,
 'loved': 10,
 'movie': 11,
 'hate': 12,
 'story': 13}

<h2>Sentence to numbers</h2>

In [13]:
sentence_to_num=[]
for sentence in tokens:
    temp=[]
    for words in sentence:
       temp.append(words_to_id[words])

    sentence_to_num.append(temp)

In [14]:
sentence_to_num

[[1, 2, 6, 11], [6, 11, 8, 4], [1, 12, 6, 11], [5, 13, 7, 9], [1, 10, 3]]

<h2>Padding</h2>

In [15]:
maxLen=len(max(sentence_to_num))

In [16]:
padded=[]
for x in sentence_to_num:
    number_of_zeros=maxLen-len(x)
    padded.append(x+[0]*number_of_zeros)

In [17]:
padded

[[1, 2, 6, 11], [6, 11, 8, 4], [1, 12, 6, 11], [5, 13, 7, 9], [1, 10, 3, 0]]

<h2>Embeddings</h2>

In [18]:
import torch

In [19]:
embedding=torch.nn.Embedding(len(vocabularySet)+1,4,padding_idx=0)

In [20]:
padded_tensor=torch.tensor(padded,dtype=torch.long)

In [21]:
padded_tensor

tensor([[ 1,  2,  6, 11],
        [ 6, 11,  8,  4],
        [ 1, 12,  6, 11],
        [ 5, 13,  7,  9],
        [ 1, 10,  3,  0]])

In [22]:
embedding.weight.shape

torch.Size([14, 4])

In [23]:
embedded=embedding(padded_tensor)

In [24]:
embedded.shape

torch.Size([5, 4, 4])

<h2>Connecting to LSTM layer</h2>

In [25]:
lstm_layer=torch.nn.LSTM(input_size=4,hidden_size=8,batch_first=True)

In [27]:
output_lstm,hiden_lstm=lstm_layer(embedded)

In [29]:
print(f'Output_Shape:{output_lstm.shape}\nHidden:{hiden_lstm[0].shape}\nCell :{hiden_lstm[1].shape}')

Output_Shape:torch.Size([5, 4, 8])
Hidden:torch.Size([1, 5, 8])
Cell :torch.Size([1, 5, 8])


In [31]:
final_hidden_state=hiden_lstm[0].squeeze(0)

In [33]:
final_hidden_state.shape

torch.Size([5, 8])

In [35]:
linear=torch.nn.Linear(8,1)

In [39]:
target=torch.tensor([1,1,0,0,1])
target.reshape(5,1)

tensor([[1],
        [1],
        [0],
        [0],
        [1]])

In [41]:
loss=torch.nn.BCEWithLogitsLoss()